<a href="https://colab.research.google.com/github/avialazraki/DI_Bootcamp/blob/main/Week16_Day1_Selenium_Crashpad_Permission_Fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 16 — Day 1 | Mini Project 1: Dynamic Web Scraping (Crashpad Permission Fix)

Scrape InMotion Hosting using Selenium and BeautifulSoup, extract displayed plan names, prices and features, and export CSV. Fixes Chrome for Testing helper executable permissions.

### 1. Install Python packages and Linux libraries required by Chrome

The default `chromium-browser` and `chromedriver` commands can be Snap wrappers on hosted notebooks. Instead, this notebook uses Google's matched Chrome for Testing binaries and installs the Linux shared libraries Chrome needs. No Snap installation is attempted.

Run this cell first. It requires a Linux environment with `apt-get` (as in Google Colab).

In [1]:
# Install only Python packages needed for this project (no blanket upgrades).
%pip -q install selenium beautifulsoup4

import os, subprocess

# Chrome for Testing needs desktop runtime libraries even in headless mode.
# Package names differ between Ubuntu/Debian releases (t64 transition).
if os.path.exists('/usr/bin/apt-get'):
    update = subprocess.run(['apt-get', 'update', '-qq'], capture_output=True, text=True)
    if update.returncode:
        print('APT update warning:', update.stderr[-1200:])

    packages = [
        'libnss3', 'libnspr4', 'libgbm1', 'libasound2t64',
        'libatk1.0-0t64', 'libatk-bridge2.0-0t64', 'libatspi2.0-0t64',
        'libcups2t64', 'libdbus-1-3', 'libdrm2', 'libxkbcommon0',
        'libxcomposite1', 'libxdamage1', 'libxfixes3', 'libxrandr2',
        'libx11-6', 'libxcb1', 'libxext6', 'libxshmfence1',
        'libpango-1.0-0', 'libcairo2', 'libglib2.0-0t64',
        'libgtk-3-0t64', 'fonts-liberation', 'ca-certificates'
    ]
    # Resolve package names from the current distro rather than assuming t64 variants.
    def available(package):
        result = subprocess.run(['apt-cache', 'show', package], capture_output=True, text=True)
        return result.returncode == 0 and 'Package:' in result.stdout

    resolved = []
    for package in packages:
        variants = [package]
        if package.endswith('t64'):
            variants.append(package[:-3])
        for candidate in variants:
            if available(candidate):
                resolved.append(candidate)
                break
        else:
            print('Not available from apt:', package)
    print('Installing Chrome shared-library dependencies:', resolved)
    installed = subprocess.run(['apt-get', 'install', '-y', '-qq', *resolved],
                               capture_output=True, text=True)
    if installed.returncode:
        print('APT install warning:', installed.stderr[-1800:])
    else:
        print('Chrome Linux library installation completed.')
else:
    print('apt-get unavailable. Ensure Chrome dependencies are installed manually.')

Installing Chrome shared-library dependencies: ['libnss3', 'libnspr4', 'libgbm1', 'libasound2t64', 'libatk1.0-0t64', 'libatk-bridge2.0-0t64', 'libatspi2.0-0t64', 'libcups2t64', 'libdbus-1-3', 'libdrm2', 'libxkbcommon0', 'libxcomposite1', 'libxdamage1', 'libxfixes3', 'libxrandr2', 'libx11-6', 'libxcb1', 'libxext6', 'libxshmfence1', 'libpango-1.0-0', 'libcairo2', 'libglib2.0-0t64', 'libgtk-3-0t64', 'fonts-liberation', 'ca-certificates']
Chrome Linux library installation completed.


In [2]:
import os, re, time, shutil, platform, subprocess
from pathlib import Path
from urllib.parse import urljoin, urlparse
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import WebDriverException, TimeoutException
print('Python:', platform.python_version())
print('Chrome:', shutil.which('google-chrome') or shutil.which('chromium') or shutil.which('chromium-browser'))

Python: 3.13.18
Chrome: None


## Browser startup: repair Chrome helper permissions

Your ChromeDriver log identified the precise issue: `posix_spawn ... chrome_crashpad_handler: Permission denied (13)`. The ZIP extractor can lose Unix executable permissions on Chrome helper binaries. This notebook repairs those permissions **on every run**, even for cached downloads, before launching Chrome. It then tries matching Chrome for Testing 131 and 120. If your runtime blocks execution for another reason, the diagnostic log will explain it.

In [3]:
import os, platform, re, shutil, subprocess, tempfile, urllib.request, zipfile
from pathlib import Path
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service

# Known Chrome for Testing versions. Each URL contains a matching browser/driver pair.
# Older versions avoid a Chrome 156 renderer crash observed in this runtime.
CHROME_VERSIONS = ['131.0.6778.85', '120.0.6099.109']
ROOT = Path('/tmp/di-cft-pinned')

def get_pair(version):
    if platform.system() != 'Linux' or platform.machine() not in ('x86_64', 'amd64'):
        raise RuntimeError('This setup is intended for Linux x86_64 Colab runtimes.')
    folder = ROOT / version
    chrome = folder / 'chrome-linux64' / 'chrome'
    chromedriver = folder / 'chromedriver-linux64' / 'chromedriver'
    for name, binary in [('chrome', chrome), ('chromedriver', chromedriver)]:
        if binary.exists():
            continue
        folder.mkdir(parents=True, exist_ok=True)
        url = f'https://storage.googleapis.com/chrome-for-testing-public/{version}/linux64/{name}-linux64.zip'
        archive = folder / f'{name}.zip'
        print('Downloading:', url)
        urllib.request.urlretrieve(url, archive)
        with zipfile.ZipFile(archive) as z:
            z.extractall(folder)
        archive.unlink(missing_ok=True)
        binary.chmod(0o755)
    # Python zipfile extraction does not reliably preserve Unix executable permissions.
    # Chrome must spawn its own crashpad handler; otherwise Chrome aborts with
    # "posix_spawn ... chrome_crashpad_handler: Permission denied (13)".
    # Apply this even when archives were downloaded during a previous run.
    chrome_dir = chrome.parent
    for helper_name in ('chrome', 'chrome_crashpad_handler', 'chrome_sandbox'):
        helper = chrome_dir / helper_name
        if helper.is_file():
            helper.chmod(helper.stat().st_mode | 0o111)
            print('Executable helper:', helper.name, oct(helper.stat().st_mode & 0o777))
    chromedriver.chmod(chromedriver.stat().st_mode | 0o111)
    return chrome, chromedriver

def check_binary(binary):
    result = subprocess.run([str(binary), '--version'], capture_output=True, text=True, timeout=15)
    return (result.stdout + result.stderr).strip(), result.returncode

def start_browser():
    failures = []
    for version in CHROME_VERSIONS:
        session = None
        try:
            print('\nTrying Chrome for Testing', version)
            chrome, chromedriver = get_pair(version)
            if shutil.which('ldd'):
                result = subprocess.run(['ldd', str(chrome)], capture_output=True, text=True)
                missing = [line.strip() for line in result.stdout.splitlines() if 'not found' in line]
                if missing:
                    raise RuntimeError('Missing Linux libraries: ' + '; '.join(missing))
            chrome_info, chrome_rc = check_binary(chrome)
            driver_info, driver_rc = check_binary(chromedriver)
            print('Browser:', chrome_info)
            print('Driver:', driver_info)
            if chrome_rc or driver_rc:
                raise RuntimeError('Chrome or ChromeDriver --version failed.')

            options = Options()
            options.binary_location = str(chrome)
            for flag in ('--headless=new', '--no-sandbox', '--disable-dev-shm-usage',
                         '--disable-gpu', '--disable-extensions', '--no-zygote',
                         '--disable-software-rasterizer', '--window-size=1280,900'):
                options.add_argument(flag)
            options.add_argument('--user-data-dir=' + tempfile.mkdtemp(prefix='di-cft-profile-'))
            log_file = ROOT / f'chromedriver-{version}.log'
            session = webdriver.Chrome(service=Service(executable_path=str(chromedriver),
                                                        log_output=str(log_file)), options=options)
            session.set_page_load_timeout(45)
            # Confirm that the browser actually renders a basic document.
            session.get('data:text/html,<html><title>Selenium OK</title><body>Ready</body></html>')
            assert session.title == 'Selenium OK', 'Browser did not render test page.'
            print('SUCCESS: Selenium Chrome session started and test page rendered.')
            return session
        except Exception as exc:
            print('Failed:', type(exc).__name__, str(exc)[:450])
            failures.append(f'{version}: {type(exc).__name__}: {str(exc)[:250]}')
            log_file = ROOT / f'chromedriver-{version}.log'
            if log_file.exists():
                # Show useful first error lines as well as last lines (not only stack frames).
                lines = log_file.read_text(errors='replace').splitlines()
                relevant = [line for line in lines if any(x in line.lower() for x in
                    ('error', 'fatal', 'crash', 'aborted', 'sandbox', 'timeout', 'failed'))]
                print('Diagnostic log:', '\n'.join(relevant[:8] + relevant[-8:])[-2500:])
            if session is not None:
                try: session.quit()
                except Exception: pass
    raise RuntimeError('No browser could start.\n' + '\n'.join(failures) +
                       '\nIf both versions fail, this runtime may not support headless Chrome. '
                       'Try a fresh Google Colab runtime or local Jupyter with Chrome installed.')

driver = start_browser()



Trying Chrome for Testing 131.0.6778.85
Downloading: https://storage.googleapis.com/chrome-for-testing-public/131.0.6778.85/linux64/chrome-linux64.zip
Downloading: https://storage.googleapis.com/chrome-for-testing-public/131.0.6778.85/linux64/chromedriver-linux64.zip
Executable helper: chrome 0o755
Executable helper: chrome_crashpad_handler 0o755
Executable helper: chrome_sandbox 0o755
Browser: Google Chrome for Testing 131.0.6778.85
Driver: ChromeDriver 131.0.6778.85 (3d81e41b6f3ac8bcae63b32e8145c9eb0cd60a2d-refs/branch-heads/6778@{#2285})
SUCCESS: Selenium Chrome session started and test page rendered.


### 3. Load dynamic pages

We start at the assigned homepage and follow a small number of same-domain hosting-plan links. We wait for JavaScript to populate the page. We do **not** bypass CAPTCHAs or access controls.

In [4]:
BASE_URL = 'https://www.inmotionhosting.com/'
MAX_PAGES = 5
WAIT_SECONDS = 20

def load_page(url):
    driver.get(url)
    WebDriverWait(driver, WAIT_SECONDS).until(
        lambda d: d.execute_script('return document.readyState') == 'complete'
    )
    time.sleep(2)  # allow client-side content to render
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    print(f'Loaded {driver.current_url} | HTML characters: {len(html):,} | title: {soup.title.get_text(" ", strip=True) if soup.title else "N/A"}')
    return soup

page_data = []
try:
    home_soup = load_page(BASE_URL)
    page_data.append((driver.current_url, home_soup))
except (TimeoutException, WebDriverException) as exc:
    print('Could not load homepage:', type(exc).__name__, str(exc)[:250])
    home_soup = None

if home_soup:
    candidate_links = []
    for a in home_soup.select('a[href]'):
        url = urljoin(BASE_URL, a.get('href', ''))
        label = a.get_text(' ', strip=True).lower()
        path = urlparse(url).path.lower()
        if (urlparse(url).hostname or '').endswith('inmotionhosting.com') and (
            any(term in path for term in ('hosting', 'vps', 'dedicated', 'wordpress')) or
            any(term in label for term in ('hosting plans', 'shared hosting', 'vps hosting'))
        ):
            url = url.split('#')[0].split('?')[0]
            if url not in candidate_links and url != BASE_URL:
                candidate_links.append(url)
    print('Candidate hosting pages:', candidate_links[:12])
    for url in candidate_links[:MAX_PAGES-1]:
        try:
            soup = load_page(url)
            page_data.append((driver.current_url, soup))
        except (TimeoutException, WebDriverException) as exc:
            print('Skipped', url, type(exc).__name__)
print('Pages inspected:', len(page_data))

Loaded https://www.inmotionhosting.com/ | HTML characters: 44,799 | title: Just a moment...
Candidate hosting pages: []
Pages inspected: 1


### 4. Inspect the HTML for hosting-plan cards

The website may use different class names over time. We inspect likely plan containers and visible price strings before extracting.

In [5]:
PRICE_RE = re.compile(r'(?:US\$|\$|USD\s*)\s*\d[\d,]*(?:\.\d{1,2})?(?:\s*/\s*(?:mo|month|yr|year))?', re.I)
CARD_SELECTORS = [
    '[class*="pricing-card"]', '[class*="pricing_card"]', '[class*="plan-card"]',
    '[class*="plan_card"]', '[class*="product-card"]', '[class*="pricing-table"]',
    '[class*="pricing-column"]', '[class*="hosting-plan"]',
    'article[class*="plan"]', 'section[class*="pricing"] article',
]
for url, soup in page_data:
    counts = {selector: len(soup.select(selector)) for selector in CARD_SELECTORS if soup.select(selector)}
    price_matches = PRICE_RE.findall(soup.get_text(' ', strip=True))
    print('\nPAGE:', url, '\nLikely cards:', counts, '\nExample price strings:', price_matches[:8])


PAGE: https://www.inmotionhosting.com/ 
Likely cards: {} 
Example price strings: []


### 5. Extract plan names, features, and prices

The extractor selects card-like blocks, reads their heading and bullet points, and stores the exact price text if found. Missing prices are left blank. A blank result means the website did not expose recognizable plan cards; it does not mean there are no plans.

In [6]:
def clean_text(s):
    return re.sub(r'\s+', ' ', s or '').strip()

def extract_cards(soup, source_url):
    rows = []
    seen_nodes = set()
    for selector in CARD_SELECTORS:
        for card in soup.select(selector):
            if id(card) in seen_nodes:
                continue
            seen_nodes.add(id(card))
            text = clean_text(card.get_text(' ', strip=True))
            if len(text) < 20 or len(text) > 3000:
                continue
            headings = card.select('h2, h3, h4, [class*="plan-name"], [class*="plan-title"]')
            name = next((clean_text(h.get_text(' ', strip=True)) for h in headings if clean_text(h.get_text(' ', strip=True))), '')
            prices = PRICE_RE.findall(text)
            features = [clean_text(li.get_text(' ', strip=True)) for li in card.select('li')]
            features = list(dict.fromkeys(x for x in features if x))[:20]
            # Ignore broad containers that represent entire page sections, not individual plans.
            if not name or (not prices and not features) or len(card.select('h2,h3,h4')) > 8:
                continue
            rows.append({
                'plan_name': name,
                'price_displayed': prices[0] if prices else None,
                'features': ' | '.join(features),
                'source_url': source_url,
            })
    return rows

all_rows = []
for url, soup in page_data:
    rows = extract_cards(soup, url)
    print(f'{url}: {len(rows)} potential plan cards')
    all_rows.extend(rows)

columns = ['plan_name', 'price_displayed', 'features', 'source_url']
plans_df = pd.DataFrame(all_rows, columns=columns).drop_duplicates(subset=columns).reset_index(drop=True)
print('Extracted records:', len(plans_df))
display(plans_df.head(20))

https://www.inmotionhosting.com/: 0 potential plan cards
Extracted records: 0


,plan_name,price_displayed,features,source_url


### 6. Validate extraction and save to CSV

Prices may be introductory, promotional, or location-dependent. Check them on the live site before drawing conclusions. If the DataFrame is empty, inspect the saved HTML and update `CARD_SELECTORS` based on the current page; do not submit invented rows.

In [7]:
OUTPUT_CSV = Path('inmotion_hosting_plans.csv')
plans_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print('Saved:', OUTPUT_CSV.resolve())
print('Rows:', len(plans_df), '| Columns:', plans_df.columns.tolist())
if plans_df.empty:
    print('WARNING: No plan cards were detected. The site may have changed its HTML, blocked automation, or rendered pricing differently.')
    if page_data:
        Path('inmotion_page_debug.html').write_text(str(page_data[-1][1]), encoding='utf-8')
        print('Saved inmotion_page_debug.html for manual selector inspection.')
else:
    print('Plans with displayed price:', plans_df['price_displayed'].notna().sum())
    print('Plans with listed features:', plans_df['features'].fillna('').ne('').sum())

Saved: /content/inmotion_hosting_plans.csv
Rows: 0 | Columns: ['plan_name', 'price_displayed', 'features', 'source_url']
Saved inmotion_page_debug.html for manual selector inspection.


### 7. Close WebDriver

Always close the browser after extraction to free system resources.

In [8]:
try:
    driver.quit()
    print('Selenium WebDriver closed.')
except NameError:
    print('WebDriver was not initialized; nothing to close.')

Selenium WebDriver closed.


### 8. Findings and limitations

- **Dynamic content:** Selenium executes JavaScript before BeautifulSoup parses the rendered HTML.
- **Data quality:** Hosting plans, prices, and features may vary by promotion, location, and date.
- **Scraping limitations:** Selectors can become outdated; automated access may be blocked. Do not circumvent restrictions.
- **Ethics:** Respect the website's terms and robots guidance, use low request volumes, and avoid collecting personal information.
- **Deliverable:** `inmotion_hosting_plans.csv` contains only information actually extracted from pages.

### Submit to GitHub / DI Learning
1. In Colab, select **Runtime → Run all** and verify the CSV contains real data.
2. Select **File → Save a copy in GitHub** (or download the `.ipynb` and push it to your repository).
3. Optionally include `inmotion_hosting_plans.csv` if it contains valid results.
4. Submit the public GitHub notebook URL to DI Learning.

If a site restriction prevents extraction, report the limitation honestly and attach your diagnostic findings.